In [0]:
# Connect to ADLS
storage_account_name = "azureetlstorage"

storage_account_key = dbutils.secrets.get(
    scope = "etl-scope",
    key   = "adls-key"
)

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

print("connected!")

In [0]:
# Read Silver enriched table
enriched_path = f"abfss://datalake@{storage_account_name}.dfs.core.windows.net/silver/orders_enriched"

enriched_df = spark.read.format("delta").load(enriched_path)

print(f"Total rows: {enriched_df.count()}")
enriched_df.printSchema()

In [0]:
from pyspark.sql import functions as F

# Gold Table 1 — Daily sales summary
daily_sales_df = enriched_df \
    .groupBy("order_date", "status") \
    .agg(
        F.count("order_id")            .alias("order_count"),
        F.sum("line_total")            .alias("gross_revenue"),
        F.avg("unit_price")            .alias("avg_order_value"),
        F.sum("quantity")              .alias("total_units_sold"),
        F.countDistinct("customer_id") .alias("unique_customers")
    ) \
    .withColumn("_gold_processed_at", F.current_timestamp())

print(f"Daily sales rows: {daily_sales_df.count()}")
daily_sales_df.show(5)

In [0]:
# Write Daily Sales to Gold
daily_sales_path = f"abfss://datalake@{storage_account_name}.dfs.core.windows.net/gold/daily_sales"

daily_sales_df.write.format("delta") \
    .mode("overwrite") \
    .partitionBy("order_date") \
    .save(daily_sales_path)

print("Gold daily sales written!")

In [0]:
# Gold Table 2 — Customer lifetime value
customer_ltv_df = enriched_df \
    .where(F.col("status") == "completed") \
    .groupBy("customer_id", "customer_name", "country", "tier") \
    .agg(
        F.count("order_id")        .alias("total_orders"),
        F.sum("line_total")        .alias("lifetime_revenue"),
        F.avg("line_total")        .alias("avg_order_value"),
        F.min("order_date")        .alias("first_order_date"),
        F.max("order_date")        .alias("last_order_date"),
        F.countDistinct("product_id").alias("unique_products_bought")
    ) \
    .withColumn(
        "days_since_last_order",
        F.datediff(F.current_date(), F.col("last_order_date"))
    ) \
    .withColumn("_gold_processed_at", F.current_timestamp())

print(f"Customer LTV rows: {customer_ltv_df.count()}")
customer_ltv_df.show(5)

In [0]:
# Write Customer LTV to Gold
customer_ltv_path = f"abfss://datalake@{storage_account_name}.dfs.core.windows.net/gold/customer_ltv"

customer_ltv_df.write.format("delta") \
    .mode("overwrite") \
    .save(customer_ltv_path)

print("Gold customer LTV written!")

In [0]:
# Gold Table 3 — Product performance by category
product_performance_df = enriched_df \
    .groupBy("category", "product_id", "product_name", "is_active") \
    .agg(
        F.count("order_id")     .alias("total_orders"),
        F.sum("quantity")       .alias("total_units_sold"),
        F.sum("line_total")     .alias("total_revenue"),
        F.avg("unit_price")     .alias("avg_selling_price"),
        F.sum(
            F.col("unit_price") - F.col("unit_cost")
        )                       .alias("total_profit"),
        F.countDistinct(
            "customer_id")      .alias("unique_customers")
    ) \
    .withColumn(
        "profit_margin_pct",
        F.round(
            F.col("total_profit") / F.col("total_revenue") * 100
        , 2)
    ) \
    .withColumn("_gold_processed_at", F.current_timestamp())

print(f"Product performance rows: {product_performance_df.count()}")
product_performance_df.show(5)

In [0]:
# Write Product Performance to Gold
product_performance_path = f"abfss://datalake@{storage_account_name}.dfs.core.windows.net/gold/product_performance"

product_performance_df.write.format("delta") \
    .mode("overwrite") \
    .partitionBy("category") \
    .save(product_performance_path)

print("Gold product performance written!")

In [0]:
# OPTIMIZE all Gold tables
spark.sql(f"OPTIMIZE delta.`{daily_sales_path}` ZORDER BY (status)")
print("Daily sales optimized!")

spark.sql(f"OPTIMIZE delta.`{customer_ltv_path}` ZORDER BY (customer_id)")
print("Customer LTV optimized!")

spark.sql(f"OPTIMIZE delta.`{product_performance_path}` ZORDER BY (product_id)")
print("Product performance optimized!")

In [0]:
from pyspark.sql import functions as F
# Read Silver reviews
silver_reviews_path = f"abfss://datalake@{storage_account_name}.dfs.core.windows.net/silver/reviews"

silver_reviews_df = spark.read.format("delta").load(silver_reviews_path)

# Calculate average rating per product
ratings_df = silver_reviews_df \
    .groupBy("product_id") \
    .agg(
        F.avg("rating")  .alias("avg_rating"),
        F.count("review_id").alias("total_reviews"),
        F.min("rating")  .alias("min_rating"),
        F.max("rating")  .alias("max_rating")
    ) \
    .withColumn("avg_rating", F.round(F.col("avg_rating"), 2))

print(f"Ratings calculated for {ratings_df.count()} products")
ratings_df.show(5)

In [0]:
import pyspark.sql.functions as F# Rebuild product performance fresh from Silver (no ratings yet)

product_performance_df = enriched_df \
    .groupBy("category", "product_id", "product_name", "is_active") \
    .agg(
        F.count("order_id")        .alias("total_orders"),
        F.sum("quantity")          .alias("total_units_sold"),
        F.sum("line_total")        .alias("total_revenue"),
        F.avg("unit_price")        .alias("avg_selling_price"),
        F.sum(
            F.col("unit_price") - F.col("unit_cost")
        )                          .alias("total_profit"),
        F.countDistinct("customer_id").alias("unique_customers")
    ) \
    .withColumn(
        "profit_margin_pct",
        F.round(
            F.col("total_profit") / F.col("total_revenue") * 100
        , 2)
    )

# Now join ratings onto fresh product performance
enriched_products_df = product_performance_df \
    .join(ratings_df, on="product_id", how="left") \
    .withColumn("_gold_processed_at", F.current_timestamp())

print(f"Enriched products: {enriched_products_df.count()} rows")
enriched_products_df.show(5)

In [0]:
# Overwrite Gold product performance with ratings included
enriched_products_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("category") \
    .save(product_performance_path)

print("Gold product performance updated with ratings!")